In [1]:
import ibis
import numpy as np
import statsmodels.api as sm

# Read data and keep the relevant columns
df = ibis.read_csv("jkp_factors.csv")
df = df.select("market", "value", "momentum", "quality")

# Convert to arrays for regression
data = df.to_pyarrow()

market = data["market"].combine_chunks().to_numpy()
value = data["value"].combine_chunks().to_numpy()
momentum = data["momentum"].combine_chunks().to_numpy()
quality = data["quality"].combine_chunks().to_numpy()

In [2]:
# 3.5
X = sm.add_constant(market)

reg_value = sm.OLS(value, X).fit()
reg_momentum = sm.OLS(momentum, X).fit()
reg_quality = sm.OLS(quality, X).fit()

beta_value = reg_value.params[1]
beta_momentum = reg_momentum.params[1]
beta_quality = reg_quality.params[1]

print("Value beta:", beta_value)
print("Momentum beta:", beta_momentum)
print("Quality beta:", beta_quality)

Value beta: -0.23868839457536156
Momentum beta: -0.2747038478367442
Quality beta: -0.0672117998039892


In [3]:
# 3.6
var_e_value = np.var(reg_value.resid, ddof=1)
var_e_momentum = np.var(reg_momentum.resid, ddof=1)
var_e_quality = np.var(reg_quality.resid, ddof=1)

print("Value residual variance:", var_e_value)
print("Momentum residual variance:", var_e_momentum)
print("Quality residual variance:", var_e_quality)

Value residual variance: 0.00142527526256314
Momentum residual variance: 0.0011549717529749804
Quality residual variance: 0.00026118000202393556


In [4]:
# 3.7
w = np.array([0.50, 0.25, 0.25])
betas = np.array([beta_value, beta_momentum, beta_quality])
res_vars = np.array([var_e_value, var_e_momentum, var_e_quality])

var_market = np.var(market, ddof=1)

beta_portfolio = w @ betas

var_portfolio = (
    beta_portfolio**2 * var_market
    + np.sum(w**2 * res_vars)
)

sd_portfolio = np.sqrt(var_portfolio)

print("Portfolio beta:", beta_portfolio)
print("Portfolio SD:", sd_portfolio)
print("Portfolio SD (%):", sd_portfolio * 100)

Portfolio beta: -0.20482310919786412
Portfolio SD: 0.023025697359763304
Portfolio SD (%): 2.30256973597633


In [5]:
# 3.8
portfolio_return = (
    0.50 * value
    + 0.25 * momentum
    + 0.25 * quality
)

sd_34 = np.std(portfolio_return, ddof=1)

print("3.4 SD (%):", sd_34 * 100)
print("3.7 SD (%):", sd_portfolio * 100)

3.4 SD (%): 1.973646570782683
3.7 SD (%): 2.30256973597633


In [6]:
residuals = np.column_stack([
    reg_value.resid,
    reg_momentum.resid,
    reg_quality.resid
])

print(np.corrcoef(residuals, rowvar=False))

[[ 1.         -0.37960419 -0.20622891]
 [-0.37960419  1.          0.18297744]
 [-0.20622891  0.18297744  1.        ]]


The estimates differ because part 3.4 utilizes the actual sample correlations among value, momentum, and quality, while 3.7 attributes their common risk to the market and assumes the remaining residual risks are uncorrelated. As shown in the data above, the residual correlations are not zero. Therefore, the two covariance structures give different portfolio risk estimates.

3.9:

I prefer the 3.4 estimate because it uses the observed covariance structure directly. Since the residuals from the market regressions remain correlated, the simplifying assumption used in 3.7 might not fully capture the relationships among the three factors.

In [8]:
from docx import Document
from docx.shared import Pt, Inches, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn



def shade_cell(cell, fill="D9EAF7"):
    tcPr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement("w:shd")
    shd.set(qn("w:fill"), fill)
    tcPr.append(shd)


def format_table(table):
    table.style = "Table Grid"
    table.alignment = WD_TABLE_ALIGNMENT.CENTER

    for row in table.rows:
        for cell in row.cells:
            cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER

            for p in cell.paragraphs:
                p.alignment = WD_ALIGN_PARAGRAPH.CENTER
                for run in p.runs:
                    run.font.name = "Times New Roman"
                    run.font.size = Pt(10)


def add_heading(doc, text):
    p = doc.add_paragraph()
    p.paragraph_format.space_before = Pt(8)
    p.paragraph_format.space_after = Pt(4)

    run = p.add_run(text)
    run.bold = True
    run.font.name = "Times New Roman"
    run.font.size = Pt(11)

    return p


def add_text(doc, text):
    p = doc.add_paragraph()
    p.paragraph_format.space_after = Pt(5)

    run = p.add_run(text)
    run.font.name = "Times New Roman"
    run.font.size = Pt(11)

    return p




doc = Document()

section = doc.sections[0]
section.top_margin = Inches(0.7)
section.bottom_margin = Inches(0.7)
section.left_margin = Inches(0.7)
section.right_margin = Inches(0.7)

style = doc.styles["Normal"]
style.font.name = "Times New Roman"
style.font.size = Pt(11)


# ---------- title ----------

p = doc.add_paragraph()
p.paragraph_format.space_after = Pt(8)

run = p.add_run("Section 3: Predicting Portfolio Risk")
run.bold = True
run.font.name = "Times New Roman"
run.font.size = Pt(14)


# ---------- 3.5 ----------

add_heading(doc, "3.5: Market betas")

add_text(
    doc,
    "Table 3 reports the market beta estimates from separate OLS regressions "
    "of Value, Momentum, and Quality on the market return."
)

table = doc.add_table(rows=2, cols=3)

headers = ["Value", "Momentum", "Quality"]
values = [beta_value, beta_momentum, beta_quality]

for j, x in enumerate(headers):
    table.cell(0, j).text = x
    shade_cell(table.cell(0, j))

for j, x in enumerate(values):
    table.cell(1, j).text = f"{x:.4f}"

format_table(table)


# ---------- 3.6 ----------

add_heading(doc, "3.6: Residual variances")

add_text(
    doc,
    "Table 4 reports the sample variance of the residuals from each of the "
    "three market regressions."
)

table = doc.add_table(rows=2, cols=3)

for j, x in enumerate(headers):
    table.cell(0, j).text = x
    shade_cell(table.cell(0, j))

residual_vars = [
    var_e_value,
    var_e_momentum,
    var_e_quality
]

for j, x in enumerate(residual_vars):
    table.cell(1, j).text = f"{x:.6f}"

format_table(table)


# ---------- 3.7 ----------

add_heading(doc, "3.7: Portfolio risk using the market model")

add_text(
    doc,
    "The Portfolio allocates 50% to Value, 25% to Momentum, and 25% to Quality. "
    "Using the market-model variance formula,"
)

p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER

run = p.add_run(
    "Var(P) = βP² Var(Market) + "
    "wV² Var(eV) + wM² Var(eM) + wQ² Var(eQ)"
)
run.italic = True
run.font.name = "Times New Roman"
run.font.size = Pt(11)

add_text(
    doc,
    f"where βP = {beta_portfolio:.4f}, the predicted standard deviation "
    f"of The Portfolio is {sd_portfolio * 100:.2f}% per month."
)


# ---------- 3.8 ----------

add_heading(doc, "3.8: Why are the two risk predictions different?")

add_text(
    doc,
    f"The sample-covariance approach in 3.4 gives a portfolio standard deviation "
    f"of {sd_34 * 100:.2f}%, while the market-model approach in 3.7 gives "
    f"{sd_portfolio * 100:.2f}%. The estimates differ because 3.4 uses the "
    "observed covariances among the three factors, whereas 3.7 attributes common "
    "risk to the market and treats the remaining residual risks as uncorrelated."
)


# ---------- 3.9 ----------

add_heading(doc, "3.9: Preferred risk prediction")

add_text(
    doc,
    "I prefer the estimate from 3.4 because it uses the observed covariance "
    "structure directly. Since the regression residuals remain correlated, "
    "the market-model assumptions in 3.7 do not fully capture the relationships "
    "among the three factors."
)


# ---------- save ----------

output_file = "Section3_PortfolioRisk_Complete.docx"
doc.save(output_file)

print("Saved:", output_file)

Saved: Section3_PortfolioRisk_Complete.docx
